# Shared-State Exploration

The current NLI backend evaluates each decision as a sequence pair:

`state + hypothesis`

Although the questions are processed in one GPU batch, the shared state is repeated for every question.

Before attempting to reuse the state representation, this notebook examines how the tokenizer and DeBERTa model combine the state and hypothesis during inference.

In [1]:
from parallel_decider import BooleanQuestion, NLIBackend

backend = NLIBackend()

backend.device

/home/mebus/workspace/parallel-decider/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

device(type='cuda')

In [2]:
state = "Inspect the repository and find the login bug."

hypothesis = "This task requires Git access."

In [3]:
inputs = backend.tokenizer(
    state,
    hypothesis,
    return_tensors="pt",
)

inputs

{'input_ids': tensor([[    1, 45295,   262, 12499,   263,   433,   262,  6607,  5554,   260,
             2,   329,  2204,  1956, 25454,   739,   260,     2]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [4]:
tokens = backend.tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

tokens

['[CLS]',
 '▁Inspect',
 '▁the',
 '▁repository',
 '▁and',
 '▁find',
 '▁the',
 '▁login',
 '▁bug',
 '.',
 '[SEP]',
 '▁This',
 '▁task',
 '▁requires',
 '▁Git',
 '▁access',
 '.',
 '[SEP]']

In [5]:
for index, token in enumerate(tokens):
    print(f"{index:>3}: {token}")

  0: [CLS]
  1: ▁Inspect
  2: ▁the
  3: ▁repository
  4: ▁and
  5: ▁find
  6: ▁the
  7: ▁login
  8: ▁bug
  9: .
 10: [SEP]
 11: ▁This
 12: ▁task
 13: ▁requires
 14: ▁Git
 15: ▁access
 16: .
 17: [SEP]


In [6]:
for key, value in inputs.items():
    print(key, value.shape)
    print(value)

input_ids torch.Size([1, 18])
tensor([[    1, 45295,   262, 12499,   263,   433,   262,  6607,  5554,   260,
             2,   329,  2204,  1956, 25454,   739,   260,     2]])
token_type_ids torch.Size([1, 18])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]])
attention_mask torch.Size([1, 18])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## Inspecting Hidden Representations

The tokenizer separates the state and hypothesis with token type IDs, but the transformer processes the combined sequence jointly.

Next, we inspect the hidden-state tensors produced by each transformer layer. This helps us understand where the state representation exists and whether any part of it could potentially be reused across questions.

In [7]:
import torch

inputs = {
    key: value.to(backend.device)
    for key, value in inputs.items()
}

with torch.inference_mode():
    outputs = backend.model(
        **inputs,
        output_hidden_states=True,
    )

In [8]:
len(outputs.hidden_states)

13

In [9]:
for index, hidden_state in enumerate(outputs.hidden_states):
    print(
        f"Layer {index:>2}: "
        f"{tuple(hidden_state.shape)}"
    )

Layer  0: (1, 18, 768)
Layer  1: (1, 18, 768)
Layer  2: (1, 18, 768)
Layer  3: (1, 18, 768)
Layer  4: (1, 18, 768)
Layer  5: (1, 18, 768)
Layer  6: (1, 18, 768)
Layer  7: (1, 18, 768)
Layer  8: (1, 18, 768)
Layer  9: (1, 18, 768)
Layer 10: (1, 18, 768)
Layer 11: (1, 18, 768)
Layer 12: (1, 18, 768)


In [10]:
final_hidden = outputs.hidden_states[-1]

final_hidden.shape

torch.Size([1, 18, 768])

In [11]:
token_types = inputs["token_type_ids"][0]

state_mask = token_types == 0
hypothesis_mask = token_types == 1

state_hidden = final_hidden[0, state_mask]
hypothesis_hidden = final_hidden[0, hypothesis_mask]

print("State:", state_hidden.shape)
print("Hypothesis:", hypothesis_hidden.shape)

State: torch.Size([11, 768])
Hypothesis: torch.Size([7, 768])


## Does the Hypothesis Change the State Representation?

The state text is identical across decisions, but the hypothesis changes.

If DeBERTa processes the pair jointly, the hidden representation of the state tokens should change depending on the hypothesis. Measuring that change helps determine whether the final state representation can be reused safely across different questions.

In [12]:
hypothesis_a = "This task requires Git access."
hypothesis_b = "This task requires browser access."

In [13]:
def get_hidden_states(
    state: str,
    hypothesis: str,
):
    encoded = backend.tokenizer(
        state,
        hypothesis,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(backend.device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        outputs = backend.model(
            **encoded,
            output_hidden_states=True,
        )

    return encoded, outputs.hidden_states

In [14]:
inputs_a, hidden_a = get_hidden_states(
    state,
    hypothesis_a,
)

inputs_b, hidden_b = get_hidden_states(
    state,
    hypothesis_b,
)

In [15]:
state_mask_a = inputs_a["token_type_ids"][0] == 0
state_mask_b = inputs_b["token_type_ids"][0] == 0

for layer_index, (layer_a, layer_b) in enumerate(
    zip(hidden_a, hidden_b)
):
    state_a = layer_a[0, state_mask_a]
    state_b = layer_b[0, state_mask_b]

    mean_absolute_difference = (
        state_a - state_b
    ).abs().mean().item()

    print(
        f"Layer {layer_index:>2}: "
        f"{mean_absolute_difference:.6f}"
    )

Layer  0: 0.000000
Layer  1: 0.023392
Layer  2: 0.031067
Layer  3: 0.039642
Layer  4: 0.050232
Layer  5: 0.050842
Layer  6: 0.049805
Layer  7: 0.058594
Layer  8: 0.078308
Layer  9: 0.072388
Layer 10: 0.093994
Layer 11: 0.100525
Layer 12: 0.308594


## Observation

The state representation is identical at layer 0, before transformer
self-attention begins.

As the input passes through the transformer layers, the hidden
representation of the state increasingly depends on the hypothesis.

The mean absolute difference grows from `0.0` at layer 0 to approximately
`0.31` at the final layer.

This confirms that the NLI model jointly encodes the state and hypothesis.
The final state representation therefore cannot be safely cached once and
reused across arbitrary questions without changing the model's computation.

A true shared-state implementation will require either:

- a different model architecture,
- a separate state/question encoder design,
- a smaller cross-attention stage, or
- an approximation that reuses early state representations.